In [6]:
import gc
import os
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import hstack as sp_hstack
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm import tqdm

# Preprocessing & Negative Sampling

## Load raw data

In [7]:
DATA_PATH = '/kaggle/input/datasets/f0rzah/kion-dataset/data_original/'
OUTPUT_DIR = '/kaggle/working/'

users_df = pd.read_csv(f'{DATA_PATH}users.csv')
items_df = pd.read_csv(f'{DATA_PATH}items.csv')
interactions_df = pd.read_csv(f'{DATA_PATH}interactions.csv', parse_dates=['last_watch_dt'])

## Cutoff and split

In [8]:
CUTOFF_DATE = pd.Timestamp('2021-05-01')
VAL_DAYS = 14
TEST_DAYS = 14

interactions_df = interactions_df[interactions_df['last_watch_dt'] >= CUTOFF_DATE].reset_index(drop=True)
interactions_df = interactions_df.sort_values('last_watch_dt')

max_date = interactions_df['last_watch_dt'].max()
test_start_date = max_date - pd.Timedelta(days=TEST_DAYS)
val_start_date = test_start_date - pd.Timedelta(days=VAL_DAYS)

In [9]:
test_df = interactions_df[interactions_df['last_watch_dt'] >= test_start_date].copy()
val_df = interactions_df[(interactions_df['last_watch_dt'] >= val_start_date) & (interactions_df['last_watch_dt'] < test_start_date)].copy()
train_df = interactions_df[interactions_df['last_watch_dt'] < val_start_date].copy()

for df in (train_df, val_df, test_df):
    df['day_of_week'] = df['last_watch_dt'].dt.dayofweek

print(f'Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}')

Train: (3059544, 6), Val: (749528, 6), Test: (888543, 6)


## Downcast

In [10]:
def downcast_dtypes(df):
    float_cols = [c for c in df if df[c].dtype == 'float64']
    int_cols = [c for c in df if df[c].dtype in ['int64', 'int32']]
    df[float_cols] = df[float_cols].astype(np.float32)
    df[int_cols] = df[int_cols].astype(np.int32)
    return df

users_df = downcast_dtypes(users_df)
items_df = downcast_dtypes(items_df)
train_df = downcast_dtypes(train_df)
val_df = downcast_dtypes(val_df)
test_df = downcast_dtypes(test_df)
gc.collect()

42

## Top-N weights

In [11]:
def calculate_top_weights(items_df, source_col, top_n):
    all_values = []
    for val in items_df[source_col].dropna():
        all_values.extend([x.strip() for x in str(val).split(',')])
    top_values = [v for v, _ in pd.Series(all_values).value_counts().head(top_n).items()]
    return {val: 1.0 / np.log2(rank + 1) for rank, val in enumerate(top_values, start=1)}

genre_weights = calculate_top_weights(items_df, 'genres', 20)
country_weights = calculate_top_weights(items_df, 'countries', 20)
actor_weights = calculate_top_weights(items_df, 'actors', 50)
director_weights = calculate_top_weights(items_df, 'directors', 50)

## Item features

In [12]:
items_features = items_df[['item_id', 'release_year', 'age_rating', 'genres', 'countries', 'actors', 'directors', 'content_type']].copy()

bins = [1890, 1989, 1999, 2009, 2019, 2030]
labels = ['<1990', '1990-1999', '2000-2009', '2010-2019', '2020+']
items_features['release_decade'] = pd.cut(items_features['release_year'], bins=bins, labels=labels, right=True).astype(str)
items_features['is_recent_release'] = (items_features['release_year'] >= 2019).astype('int8')

age_map = {0.0: 0, 6.0: 1, 12.0: 2, 16.0: 3, 18.0: 4, 21.0: 5}
items_features['age_rating_ord'] = items_features['age_rating'].map(age_map).fillna(2).astype('int8')
items_features['is_kids_genre'] = items_features['genres'].apply(lambda x: 1 if pd.notna(x) and 'для детей' in str(x).lower() else 0).astype('int8')

In [13]:
def apply_weights(source_col, weights_dict):
    return items_features[source_col].apply(
        lambda x: sum(weights_dict.get(item.strip(), 0.0) for item in str(x).split(',')) if pd.notna(x) else 0.0
    )

items_features['genre_score'] = apply_weights('genres', genre_weights).astype('float32')
items_features['country_score'] = apply_weights('countries', country_weights).astype('float32')
items_features['actor_score'] = apply_weights('actors', actor_weights).astype('float32')
items_features['director_score'] = apply_weights('directors', director_weights).astype('float32')

In [14]:
item_stats = train_df.groupby('item_id').agg(
    item_popularity_log=('user_id', lambda x: np.log1p(len(x))),
    item_avg_duration_log=('total_dur', lambda x: np.log1p(x.mean()))
).reset_index()

items_features = items_features.merge(item_stats, on='item_id', how='left')
items_features['item_popularity_log'] = items_features['item_popularity_log'].fillna(0.0).astype('float32')
items_features['item_avg_duration_log'] = items_features['item_avg_duration_log'].fillna(0.0).astype('float32')

items_features = items_features.drop(columns=['release_year', 'age_rating', 'genres', 'countries', 'actors', 'directors'])
print(f'items_features: {items_features.shape}')

items_features: (15963, 12)


## User features

In [15]:
users_features = users_df[['user_id', 'age', 'sex', 'income', 'kids_flg']].copy()
users_features['age'] = users_features['age'].fillna('Unknown').astype(str)
users_features['sex'] = users_features['sex'].fillna('Unknown').astype(str)
users_features['income'] = users_features['income'].fillna('Unknown').astype(str)
users_features['kids_flg'] = users_features['kids_flg'].fillna(0).astype('int8')

In [16]:
user_stats = train_df.groupby('user_id').agg(
    user_total_interactions=('item_id', 'count'),
    user_avg_duration=('total_dur', 'mean')
).reset_index()

users_features = users_features.merge(user_stats, on='user_id', how='left')
users_features['user_total_interactions'] = users_features['user_total_interactions'].fillna(0).astype('int32')
users_features['user_avg_duration'] = users_features['user_avg_duration'].fillna(0.0).astype('float32')
users_features['user_total_interactions_log'] = np.log1p(users_features['user_total_interactions']).astype('float32')
users_features['user_avg_duration_log'] = np.log1p(users_features['user_avg_duration']).astype('float32')
print(f'users_features: {users_features.shape}')

users_features: (840197, 9)


## Build pairs

In [17]:
def build_pairs(user_item_df, users_features, items_features, target):
    df = user_item_df.copy()
    df = df.merge(users_features, on='user_id', how='left')
    df = df.merge(items_features, on='item_id', how='left')
    df['is_kids_user_and_kids_content'] = ((df['kids_flg'] == 1) & (df['age_rating_ord'] <= 1)).astype('int8')
    df['target'] = target
    return df

## Negative sampling

In [18]:
N_NEGATIVES = 2
rng = np.random.default_rng(42)

pos_pairs = set(zip(train_df['user_id'], train_df['item_id']))
users_pool = train_df['user_id'].unique()
items_pool = items_features['item_id'].unique()

In [19]:
needed = len(train_df) * N_NEGATIVES
batch_size = needed * 2
neg_rows = []

while len(neg_rows) < needed:
    rand_u = rng.choice(users_pool, batch_size)
    rand_i = rng.choice(items_pool, batch_size)
    valid = [(u, i) for u, i in zip(rand_u, rand_i) if (u, i) not in pos_pairs]
    neg_rows.extend(valid)

neg_rows = neg_rows[:needed]
negatives_df = pd.DataFrame(neg_rows, columns=['user_id', 'item_id'])
negatives_df['user_id'] = negatives_df['user_id'].astype('int32')
negatives_df['item_id'] = negatives_df['item_id'].astype('int32')
print(f'Negatives: {negatives_df.shape}')

Negatives: (6119088, 2)


## Assemble train_final

In [20]:
positives_train = build_pairs(
    train_df[['user_id', 'item_id', 'watched_pct', 'day_of_week']],
    users_features, items_features, target=1
)
negatives_train = build_pairs(
    negatives_df[['user_id', 'item_id']],
    users_features, items_features, target=0
)

train_final = pd.concat([positives_train, negatives_train], ignore_index=True)
train_final = train_final.sample(frac=1, random_state=42).reset_index(drop=True)

cat_cols = ['content_type', 'release_decade', 'age', 'sex', 'income']
for col in cat_cols:
    train_final[col] = train_final[col].astype(str)

del positives_train, negatives_train, neg_rows, negatives_df
gc.collect()
print(f'train_final: {train_final.shape}')
print(train_final['target'].value_counts().to_dict())

train_final: (9178632, 25)
{0: 6119088, 1: 3059544}


## Cold / warm users

In [21]:
COLD_THRESHOLD = 5

user_counts_train = train_df.groupby('user_id').size()
cold_users = user_counts_train[user_counts_train < COLD_THRESHOLD].index.tolist()
warm_users = user_counts_train[user_counts_train >= COLD_THRESHOLD].index.tolist()
print(f'Cold: {len(cold_users)}, Warm: {len(warm_users)}')

Cold: 452500, Warm: 181139


## Candidate retrieval

In [22]:
knn_items = items_df[['item_id', 'genres', 'countries', 'actors', 'directors']].copy()
for col in ['genres', 'countries', 'actors', 'directors']:
    knn_items[col] = knn_items[col].fillna('').apply(lambda x: [i.strip() for i in str(x).split(',') if i.strip()])

feature_matrix = sp_hstack([
    MultiLabelBinarizer(sparse_output=True).fit_transform(knn_items['genres']) * 1.0,
    MultiLabelBinarizer(sparse_output=True).fit_transform(knn_items['countries']) * 1.0,
    MultiLabelBinarizer(sparse_output=True).fit_transform(knn_items['actors']) * 0.5,
    MultiLabelBinarizer(sparse_output=True).fit_transform(knn_items['directors']) * 0.5,
], format='csr')

item_ids = knn_items['item_id'].tolist()
item_id_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}
print(f'feature_matrix: {feature_matrix.shape}')

feature_matrix: (15963, 70907)


In [23]:
user_history = train_df.groupby('user_id')['item_id'].apply(list).to_dict()
top_500_popular = train_df['item_id'].value_counts().head(500).index.tolist()

def get_candidates_for_user(user_id, is_cold, top_k_items=250):
    if is_cold or user_id not in user_history or not user_history[user_id]:
        return top_500_popular

    history = user_history[user_id]
    history_indices = [item_id_to_idx[i] for i in history if i in item_id_to_idx]
    if not history_indices:
        return top_500_popular

    sims = cosine_similarity(feature_matrix[history_indices], feature_matrix)
    top_idx = np.argsort(sims.mean(axis=0))[::-1][:top_k_items]
    similar_items = [item_ids[idx] for idx in top_idx]

    candidates = list(dict.fromkeys(similar_items + top_500_popular))
    history_set = set(history)
    candidates = [i for i in candidates if i not in history_set]
    return candidates[:500]

## Evaluation samples

In [24]:
SAMPLE_SIZE = 500

user_counts_train = train_df.groupby('user_id').size()
is_cold_train = user_counts_train < COLD_THRESHOLD
cold_pool_train = user_counts_train[is_cold_train].index.tolist()
warm_pool_train = user_counts_train[~is_cold_train].index.tolist()
cold_ratio_train = len(cold_pool_train) / (len(cold_pool_train) + len(warm_pool_train))
n_cold_train = int(SAMPLE_SIZE * cold_ratio_train)
n_warm_train = SAMPLE_SIZE - n_cold_train

np.random.seed(42)
train_sampled_users = np.concatenate([
    np.random.choice(cold_pool_train, size=n_cold_train, replace=False),
    np.random.choice(warm_pool_train, size=n_warm_train, replace=False),
])

user_counts_val = val_df.groupby('user_id').size()
is_cold_val = user_counts_val < COLD_THRESHOLD
cold_pool_val = user_counts_val[is_cold_val].index.tolist()
warm_pool_val = user_counts_val[~is_cold_val].index.tolist()
cold_ratio_val = len(cold_pool_val) / (len(cold_pool_val) + len(warm_pool_val))
n_cold_val = int(SAMPLE_SIZE * cold_ratio_val)
n_warm_val = SAMPLE_SIZE - n_cold_val

np.random.seed(42)
val_sampled_users = np.concatenate([
    np.random.choice(cold_pool_val, size=n_cold_val, replace=False),
    np.random.choice(warm_pool_val, size=n_warm_val, replace=False),
])

In [25]:
train_ground_truth = train_df[train_df['user_id'].isin(train_sampled_users)].groupby('user_id')['item_id'].apply(list).to_dict()
val_ground_truth = val_df[val_df['user_id'].isin(val_sampled_users)].groupby('user_id')['item_id'].apply(list).to_dict()

In [26]:
train_candidates = {}
for uid in tqdm(train_sampled_users, desc='Train candidates'):
    train_candidates[int(uid)] = get_candidates_for_user(uid, uid in cold_users)

val_candidates = {}
for uid in tqdm(val_sampled_users, desc='Val candidates'):
    val_candidates[int(uid)] = get_candidates_for_user(uid, uid in cold_users)

Val candidates: 100%|██████████| 500/500 [00:13<00:00, 36.86it/s]


In [27]:
eval_data = {
    'train_sampled_users': train_sampled_users,
    'val_sampled_users': val_sampled_users,
    'train_ground_truth': train_ground_truth,
    'val_ground_truth': val_ground_truth,
    'train_candidates': train_candidates,
    'val_candidates': val_candidates,
    'cold_users': cold_users,
    'warm_users': warm_users,
}

with open(f'{OUTPUT_DIR}evaluation_samples.pkl', 'wb') as f:
    pickle.dump(eval_data, f)
print('evaluation_samples.pkl saved')

evaluation_samples.pkl saved


## Save artifacts

In [28]:
train_final.to_parquet(f'{OUTPUT_DIR}train_final.parquet', index=False)
items_features.to_parquet(f'{OUTPUT_DIR}items_features.parquet', index=False)
users_features.to_parquet(f'{OUTPUT_DIR}users_features.parquet', index=False)

print(f'train_final: {train_final.shape}')
print(f'items_features: {items_features.shape}')
print(f'users_features: {users_features.shape}')

train_final: (9178632, 25)
items_features: (15963, 12)
users_features: (840197, 9)


In [29]:
knn_artifacts = {
    'feature_matrix': feature_matrix,
    'item_ids': item_ids,
    'item_id_to_idx': item_id_to_idx,
    'user_history': user_history,
    'top_500_popular': top_500_popular,
    'cold_users': cold_users,
}

with open(f'{OUTPUT_DIR}knn_artifacts.pkl', 'wb') as f:
    pickle.dump(knn_artifacts, f)
print('knn_artifacts.pkl saved')

knn_artifacts.pkl saved
